<a href="https://colab.research.google.com/github/Ramanathan267/ITA1414-Ethical-hacking-project-/blob/main/imp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import hashlib
import datetime

# =========================================================
# 1. DATABASE LAYER (Simulated with Dictionaries)
# =========================================================
class SecurityDatabase:
    def __init__(self):
        self.role_permissions = {
            "ADMIN": ["CREATE", "READ", "UPDATE", "DELETE", "VIEW_LOGS"],
            "MANAGER": ["READ", "UPDATE", "CREATE"],
            "EMPLOYEE": ["READ"]
        }

        self.users = {
            "admin_user": {
                "password": self._hash_password("admin@123"),
                "role": "ADMIN"
            },
            "manager_user": {
                "password": self._hash_password("manager@456"),
                "role": "MANAGER"
            },
            "staff_user": {
                "password": self._hash_password("staff@789"),
                "role": "EMPLOYEE"
            }
        }

    def _hash_password(self, password):
        return hashlib.sha256(password.encode()).hexdigest()

    def authenticate(self, username, password):
        if username in self.users:
            if self.users[username]["password"] == self._hash_password(password):
                return self.users[username]["role"]
        return None

# =========================================================
# 2. CORE SECURITY ENGINE (The RBAC Middleware)
# =========================================================
class RBACEngine:
    def __init__(self, db):
        self.db = db
        self.current_user = None
        self.current_role = None

    def login(self):
        print("\n" + "="*30)
        print("   SECURE SYSTEM LOGIN")
        print("="*30)
        user = input("Username: ")
        pwd = input("Password: ")

        role = self.db.authenticate(user, pwd)
        if role:
            self.current_user = user
            self.current_role = role
            print(f"\n[+] Welcome {user}! Logged in with {role} privileges.")
            return True
        else:
            print("\n[!] AUTHENTICATION FAILED: Invalid credentials.")
            return False

    def authorize(self, action):
        """Checks if the logged-in user can perform the requested action."""
        allowed_actions = self.db.role_permissions.get(self.current_role, [])
        if action in allowed_actions:
            self.log_event(action, "GRANTED")
            return True
        else:
            self.log_event(action, "DENIED")
            return False

    def log_event(self, action, status):
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{timestamp}] USER: {self.current_user} | ROLE: {self.current_role} | ACTION: {action} | STATUS: {status}"
        # In a real system, this would write to a .log file
        with open("audit_trails.log", "a") as f:
            f.write(log_entry + "\n")

# =========================================================
# 3. APPLICATION LAYER (Simulated File System)
# =========================================================
def main_application():
    db = SecurityDatabase()
    engine = RBACEngine(db)

    if not engine.login():
        return

    while True:
        print("\n--- Available Operations ---")
        print("1. READ File\n2. CREATE/UPDATE File\n3. DELETE File\n4. VIEW AUDIT LOGS\n5. LOGOUT")
        choice = input("\nSelect an operation (1-5): ")

        if choice == '1':
            if engine.authorize("READ"):
                print("SUCCESS: File content displayed.")
            else:
                print("ACCESS DENIED: You do not have 'READ' permission.")

        elif choice == '2':
            if engine.authorize("UPDATE"):
                print("SUCCESS: File has been updated.")
            else:
                print("ACCESS DENIED: You do not have 'UPDATE' permission.")

        elif choice == '3':
            if engine.authorize("DELETE"):
                print("SUCCESS: File permanently deleted from system.")
            else:
                print("ACCESS DENIED: Higher privileges (ADMIN) required for deletion.")

        elif choice == '4':
            if engine.authorize("VIEW_LOGS"):
                print("\n--- SYSTEM AUDIT TRAILS ---")
                try:
                    with open("audit_trails.log", "r") as f:
                        print(f.read())
                except FileNotFoundError:
                    print("No logs found.")
            else:
                print("ACCESS DENIED: Only Administrators can view security logs.")

        elif choice == '5':
            print("Logging out safely...")
            break
        else:
            print("Invalid choice.")

if __name__ == "__main__":
    main_application()


   SECURE SYSTEM LOGIN

[+] Welcome admin_user! Logged in with ADMIN privileges.

--- Available Operations ---
1. READ File
2. CREATE/UPDATE File
3. DELETE File
4. VIEW AUDIT LOGS
5. LOGOUT
SUCCESS: File permanently deleted from system.

--- Available Operations ---
1. READ File
2. CREATE/UPDATE File
3. DELETE File
4. VIEW AUDIT LOGS
5. LOGOUT
SUCCESS: File permanently deleted from system.

--- Available Operations ---
1. READ File
2. CREATE/UPDATE File
3. DELETE File
4. VIEW AUDIT LOGS
5. LOGOUT

--- SYSTEM AUDIT TRAILS ---
[2026-03-04 02:52:43] USER: admin_user | ROLE: ADMIN | ACTION: DELETE | STATUS: GRANTED
[2026-03-04 02:52:51] USER: admin_user | ROLE: ADMIN | ACTION: DELETE | STATUS: GRANTED
[2026-03-04 02:53:07] USER: admin_user | ROLE: ADMIN | ACTION: VIEW_LOGS | STATUS: GRANTED


--- Available Operations ---
1. READ File
2. CREATE/UPDATE File
3. DELETE File
4. VIEW AUDIT LOGS
5. LOGOUT
